# Response Evaluator Guardrail

**What is response evaluation?** Using a second LLM to check an LLM's answer, for example whether it actually answers the user's question.

**How this guardrail helps:** a RAG pipeline can return text that sounds fine but doesn't answer the question or gets it wrong. The [Response Evaluator](https://guardrailsai.com/hub/validator/guardrails/response_evaluator) validator asks a second LLM a yes/no question about the answer before it reaches the user. `Yes` passes and `No` fails.

**One-time setup** (already done in `myenv`, the kernel for this notebook):
```bash
pip install guardrails-ai guardrails-ai-response-evaluator
```

It also needs an API key for the LLM provider. This notebook uses Groq, so `GROQ_API_KEY` must be set in `.env`.

In [ ]:
import warnings

# silence harmless warnings from tqdm and guardrails' sync fallback in Jupyter
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message="Could not obtain an event loop")

from dotenv import load_dotenv
from guardrails import Guard  ## the main wrapper for Guardrails
from guardrails.errors import ValidationError
from guardrails_ai.response_evaluator import ResponseEvaluator

# load the env vars
load_dotenv()

## 1. Create the validator and the guard

`llm_callable` is a [LiteLLM](https://docs.litellm.ai/docs/providers) model name, so the `groq/` prefix selects the provider. `on_fail="noop"` records the failure but leaves the text unchanged.

In [ ]:
evaluator_model = "groq/openai/gpt-oss-120b"

# create the validator
validator = ResponseEvaluator(llm_callable=evaluator_model, on_fail="noop")

# create guard and add the validator to the guard
guard = Guard().use(validator)
guard.configure(allow_metrics_collection=False)

## 2. Validate LLM responses

Pass the check as `validation_question` in `metadata`. Word it so that `Yes` means the response is good.

In [ ]:
user_query = "What is the capital of India?"
validation_question = f"Does this response correctly answer the question: {user_query}"

llm_response_1 = "The capital of India is New Delhi."
llm_response_2 = "India is known for its rich history and culture. It has a city with large metro rail network and is one of the most populous cities in the world."

response_1 = guard.validate(llm_response_1, metadata={"validation_question": validation_question})

print(f"Validation Result: {response_1.validation_passed}")
print(f"Raw LLM Response: {response_1.raw_llm_output}")
print(f"Validated Output: {response_1.validated_output}")

With `noop`, `validated_output` still contains the failed text, so check `validation_passed` before showing it to a user.

In [ ]:
response_2 = guard.validate(llm_response_2, metadata={"validation_question": validation_question})

print(f"Validation Result: {response_2.validation_passed}")
print(f"Raw LLM Response: {response_2.raw_llm_output}")
print(f"Validated Output: {response_2.validated_output}")
print(f"Failure Reason: {response_2.validation_summaries[0].failure_reason}")

## 3. `on_fail="exception"` — block a bad answer

In [ ]:
exception_guard = Guard().use(ResponseEvaluator(llm_callable=evaluator_model, on_fail="exception"))
exception_guard.configure(allow_metrics_collection=False)

try:
    exception_guard.validate("The capital of India is Mumbai.", metadata={"validation_question": validation_question})
except ValidationError as e:
    print(f"Blocked: {e}")

## 4. Things to keep in mind

- **Cost and data:** each check is an extra LLM call, which adds cost and latency and sends the response to the provider.
- **It's only as good as the evaluator:** the second LLM can be wrong too, so it isn't proof that an answer is correct.
- **Replies must be exactly yes or no:** anything else, even `Yes.` with a full stop, fails validation unless you pass `"pass_on_invalid": True` in `metadata`.